# importing and loading data

In [ ]:
%pip install pandas tqdm openai python-dotenv

In [ ]:
import pandas as pd
import json
import time
from tqdm import tqdm
from openai import OpenAI
import os
from dotenv import load_dotenv

* og data named saudi job market
* data after cleaning named CleanedData(allsteps)





In [ ]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

* code for descovering categories

In [ ]:
df = pd.read_excel("CleanedData(AllSteps).xlsx")

df["job_title"] = df["job_title"].fillna("")
df["job_description"] = df["job_description"].fillna("")


In [ ]:
sample_df = df.sample(
    n=100,
    random_state=42
).copy()

# Keep only useful fields
sample_jobs = sample_df[["job_title", "job_description"]].copy()

# Shorten descriptions to avoid huge prompt
sample_jobs["description_short"] = (
    sample_jobs["job_description"]
    .astype(str)
    .str.slice(0, 700)
)

# Convert sample to compact text
jobs_text = ""

for i, row in sample_jobs.iterrows():
    jobs_text += f"""
Job {i}
Title: {row["job_title"]}
Description: {row["description_short"]}
---
"""




#Category discovery prompt


DISCOVERY_PROMPT = """
You are an expert labor-market analyst and job taxonomy designer.

You will receive a sample of 500 job postings.

Your task is to discover a practical category schema for job classification.

Requirements:
- Create 10 to 15 high-level job categories.
- Ignore company descriptions, company history, benefits, legal notices, recruitment fraud warnings, diversity statements, and marketing content.
- Categories must be broad enough for machine learning classification.
- Categories must not be too specific, like "Oracle Developer" or "HVAC Engineer".
- Categories must not overlap too much.
- Categories should fit jobs from technology, engineering, business, sales, finance, HR, operations, healthcare, education, and legal.
- Avoid duplicate categories with similar meanings.
- Include "Other" as the final fallback category.
- For each category, provide:
  1. category_name
  2. short_definition
  3. example_job_titles
  4. boundary_notes explaining what belongs and what should go elsewhere.

Return ONLY valid JSON using this structure:

{
  "categories": [
    {
      "category_name": "",
      "short_definition": "",
      "example_job_titles": [],
      "boundary_notes": ""
    }
  ]
}
"""


# Ask LLM to discover categories


response = client.responses.create(
    model="gpt-5-mini",
    input=[
        {
            "role": "system",
            "content": DISCOVERY_PROMPT
        },
        {
            "role": "user",
            "content": jobs_text
        }
    ]
)

result_text = response.output_text.strip()

print(result_text)


# Parse JSON


category_schema = json.loads(result_text)

categories_df = pd.DataFrame(category_schema["categories"])

categories_df

In [ ]:
categories_df.to_csv(
    "jobs_LLM_categories.csv",
    index=False,
    encoding="utf-8-sig"
)

* labeling after extracting categories using LLM

In [ ]:
# Categories Extracted

VALID_CATEGORIES = [
    "Software & Application Engineering",
    "AI/ML",
    "Data Analytics & BI",
    "IT Infrastructure, Cloud & DevOps",
    "Cybersecurity, Risk & Security Operations",
    "Industrial, Process & Energy Engineering",
    "Built Environment & Construction Engineering",
    "Project & Program Management / Project Controls",
    "Product, Design & User Experience",
    "Sales, Pre-Sales & Account Management",
    "Finance & Accounting",
    "Procurement, Supply Chain & Logistics",
    "Operations & Facilities",
    "Human Resources, Learning & People Operations",
    "Marketing, Communications & Content",
    "Healthcare, Clinical & Life Sciences",
    "Legal, Risk & Compliance",
    "Education, Training & Research",
    "Other"
]

In [ ]:
# Prompt


SYSTEM_PROMPT = """
You are an expert job classification and skills extraction system.

Analyze the job title and job description.

IMPORTANT RULES:
- Ignore company descriptions, company history, benefits, legal notices, recruitment fraud warnings, diversity statements, and marketing content.
- Focus only on job responsibilities, requirements, qualifications, tools, technologies, certifications, and experience.
- Use the job title as a strong signal, but verify using the description.
- If the title is ambiguous, rely mainly on responsibilities and requirements.
- Choose exactly ONE category from the valid categories.
- Do not create new categories.
- Do not modify category names.
- Extract only skills explicitly mentioned or strongly implied.
- Prefer specific technologies, tools, software, frameworks, databases, cloud platforms, programming languages, engineering applications, methodologies, and certifications.
- Only return broader domain skills when no specific technologies or tools are mentioned.
- Avoid responsibilities, duties, generic soft skills, leadership traits, teamwork, communication, coordination, and presentation skills.
- Return JSON only.

Valid Categories:
- Software & Application Engineering
- Data Analytics & BI
- AI/ML
- IT Infrastructure, Cloud & DevOps
- Cybersecurity, Risk & Security Operations
- Industrial, Process & Energy Engineering
- Built Environment & Construction Engineering
- Project & Program Management / Project Controls
- Product, Design & User Experience
- Sales, Pre-Sales & Account Management
- Finance & Accounting
- Procurement, Supply Chain & Logistics
- Operations & Facilities
- Human Resources, Learning & People Operations
- Marketing, Communications & Content
- Healthcare, Clinical & Life Sciences
- Legal, Risk & Compliance
- Education, Training & Research
- Other

Category Guidance:

Software & Application Engineering:
Software Engineer, Backend Developer, Frontend Developer, Full Stack Developer, Mobile Developer, QA Engineer, Test Automation Engineer, Oracle APEX Developer, Application Developer.

Data Analytics & BI:
Data Analyst, BI Analyst, Reporting Analyst, Power BI Developer, Dashboard Developer, Data Visualization Specialist

AI/ML:
AI Engineer, Machine Learning Engineer, ML Scientist, NLP Engineer, Computer Vision Engineer, AI Researcher

IT Infrastructure, Cloud & DevOps:
IT Engineer, Systems Engineer, Network Engineer, System Administrator, Technical Support Engineer, Field Service Engineer, Cloud Engineer, DevOps Engineer, Site Reliability Engineer, Platform Engineer.

Cybersecurity, Risk & Security Operations:
Cybersecurity Analyst, Security Engineer, SOC Analyst, GRC Specialist, Information Security Officer, Security Architect, Penetration Tester, Security Controls Advisor.

Industrial, Process & Energy Engineering:
Mechanical Engineer, Electrical Engineer, Process Engineer, Instrumentation Engineer, Control Systems Engineer, Chemical Engineer, Reliability Engineer, Rotating Equipment Engineer, Flare & Relief Systems Engineer, Energy Engineer.

Built Environment & Construction Engineering:
Civil Engineer, Structural Engineer, Architectural Engineer, Site Engineer, Resident Engineer, Construction Engineer, Quantity Surveyor, Planning Engineer in construction projects.

Project & Program Management / Project Controls:
Project Manager, Program Manager, PMO Specialist, Project Coordinator, Project Controls Manager, Project Controls Specialist, Planning & Controls Manager.

Product, Design & User Experience:
Product Manager, Product Owner, UX Designer, UI Designer, Service Designer, Solution Consultant, Solution Architect, Technical Architect.

Sales, Pre-Sales & Account Management:
Sales Executive, Account Manager, Business Development Manager, Pre-Sales Engineer, Sales Engineer, Key Account Manager, Client Partner.

Finance & Accounting:
Accountant, Auditor, Financial Analyst, Project Accountant, Tax Specialist, Investment Analyst, Treasury Analyst, Cost Accountant.

Procurement, Supply Chain & Logistics:
Procurement Specialist, Buyer, Supply Chain Specialist, Logistics Coordinator, Warehouse Manager, Inventory Planner, Demand Planner, Order Fulfillment Specialist.

Operations & Facilities:
Operations Manager, Operations Specialist, Facility Manager, Maintenance Supervisor, Production Supervisor, Plant Operator, Service Operations Coordinator.

Human Resources, Learning & People Operations:
HR Specialist, Recruiter, Talent Acquisition Specialist, Learning & Development Specialist, Compensation & Benefits Specialist, HR Business Partner.

Marketing, Communications & Content:
Marketing Specialist, Digital Marketing Specialist, Content Creator, Communications Specialist, Social Media Specialist, Brand Manager, PR Specialist.

Healthcare, Clinical & Life Sciences:
Doctor, Nurse, Pharmacist, Clinical Specialist, Medical Representative, Laboratory Technician, Biomedical Scientist.

Legal, Risk & Compliance:
Lawyer, Legal Counsel, Compliance Officer, Risk Analyst, Regulatory Affairs Specialist, Contract Specialist.

Education, Training & Research:
Teacher, Lecturer, Trainer, Instructor, Academic Researcher, Training Specialist.

Other:
Use only when the job does not fit any listed category.

Return exactly this JSON structure:

{
  "job_category": "",
  "skills": []
}

Skill Extraction Rules:
- Return 3 to 8 skills.
- Use clean skill names, not full sentences.
- Skills should look like resume/profile skills.
- Good examples:
  - Python
  - SQL
  - Power BI
  - Tableau
  - AWS
  - Docker
  - Kubernetes
  - SAP
  - Oracle
  - AutoCAD
  - Primavera P6
  - Aspen HYSYS
  - ISO 27001
  - CISSP
  - Project Controls
  - Financial Analysis
  - Procurement
  - Lean Manufacturing
"""

In [ ]:
# Labeling function


def label_job(job_title, description):
    user_prompt = f"""
Job Title:
{job_title}

Job Description:
{description}
"""

    try:
        response = client.responses.create(
            model="gpt-5-mini",
            input=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ]
        )

        result_text = response.output_text.strip()
        result = json.loads(result_text)

        category = result.get("job_category", "Other")
        skills = result.get("skills", [])

        if category not in VALID_CATEGORIES:
            category = "Other"

        if not isinstance(skills, list):
            skills = []

        skills = [
            str(skill).strip()
            for skill in skills
            if str(skill).strip()
        ]

        skills = skills[:8]

        return category, skills

    except Exception as e:
        print("Error:", e)
        return "Other", []

In [ ]:
test_df = df.sample(10, random_state=7)

for _, row in test_df.iterrows():
    category, skills = label_job(
        row["job_title"],
        row["job_description"]
    )

    print(row["job_title"])
    print("Category:", category)
    print("Skills:", skills)
    print("-" * 50)

### label 5k sample

In [ ]:
# Shuffle once
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

sample_1kA = df.iloc[0:1000]
sample_1kB = df.iloc[1000:2000]
sample_1kC = df.iloc[2000:3000]
sample_1kD = df.iloc[3000:4000]
sample_1kE = df.iloc[4000:5000]

In [ ]:
tqdm.pandas()

sample_1kA[["job_category", "skills"]] = sample_1kA.progress_apply(
    lambda row: pd.Series(
        label_job(
            row["job_title"],
            row["job_description"]
        )
    ),
    axis=1
)

# Convert skills list to comma-separated string
sample_1kA["skills"] = sample_1kA["skills"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")



sample_1kA.to_csv(
    "jobs_labeled_with_skills_A.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File saved")

In [ ]:
tqdm.pandas()

sample_1kB[["job_category", "skills"]] = sample_1kB.progress_apply(
    lambda row: pd.Series(
        label_job(
            row["job_title"],
            row["job_description"]
        )
    ),
    axis=1
)

# Convert skills list to comma-separated string
sample_1kB["skills"] = sample_1kB["skills"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")



sample_1kB.to_csv(
    "jobs_labeled_with_skills_B.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File saved")

In [ ]:
tqdm.pandas()

sample_1kC[["job_category", "skills"]] = sample_1kC.progress_apply(
    lambda row: pd.Series(
        label_job(
            row["job_title"],
            row["job_description"]
        )
    ),
    axis=1
)

# Convert skills list to comma-separated string
sample_1kC["skills"] = sample_1kC["skills"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")



sample_1kC.to_csv(
    "jobs_labeled_with_skills_C.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File saved")

In [ ]:
tqdm.pandas()

sample_1kD[["job_category", "skills"]] = sample_1kD.progress_apply(
    lambda row: pd.Series(
        label_job(
            row["job_title"],
            row["job_description"]
        )
    ),
    axis=1
)

# Convert skills list to comma-separated string
sample_1kD["skills"] = sample_1kD["skills"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")



sample_1kD.to_csv(
    "jobs_labeled_with_skills_D.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File saved")

In [ ]:
#not needed 
tqdm.pandas()

sample_1kE[["job_category", "skills"]] = sample_1kE.progress_apply(
    lambda row: pd.Series(
        label_job(
            row["job_title"],
            row["job_description"]
        )
    ),
    axis=1
)

# Convert skills list to comma-separated string
sample_1kE["skills"] = sample_1kE["skills"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")


sample_1kE.to_csv(
    "jobs_labeled_with_skills_E.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File saved")

* final labeld dataset

In [ ]:
a = pd.read_csv('jobs_labeled_with_skills_A.csv')
b = pd.read_csv('jobs_labeled_with_skills_B.csv')
c = pd.read_csv('jobs_labeled_with_skills_C.csv')
d = pd.read_csv('jobs_labeled_with_skills_D.csv')

final = pd.concat([a, b, c, d])

final.to_csv(
    "jobs_labeled_with_skills_4k.csv",
    index=False,
    encoding="utf-8-sig"
)